# Clinical-Grade CXR Report Generation: Fine-Tuning PaliGemma on MIMIC-CXR

This notebook builds an end-to-end multimodal training pipeline for chest X-ray report generation using **QLoRA** on **google/paligemma-3b-pt-224**.

Clinical framing:
- We explicitly parse and train on **Findings** and **Impression** because they represent the most clinically actionable and standardized report components.
- We use quantization + LoRA to reduce memory footprint, enabling training on commodity GPUs while preserving adaptation quality.

Assumptions:
- You have access to MIMIC-CXR-JPG files and metadata/report CSVs.
- You run on a GPU notebook with at least ~16GB VRAM.

## Phase 1: Environment & MIMIC-CXR Data Orchestration

In [ ]:
# Code Cell 1: Install required libraries
# In Kaggle/Colab, run once per fresh runtime.

!pip -q install -U bitsandbytes peft accelerate transformers datasets evaluate nltk rouge-score sentencepiece

In [ ]:
# Code Cell 2: Robust MIMIC-CXR data loader + report section extraction

import os
import re
from pathlib import Path
from typing import Optional, Tuple

import numpy as np
import pandas as pd
from PIL import Image

# -----------------------------
# User-configurable paths
# -----------------------------
# Example layout root where you placed MIMIC-CXR-JPG files:
# /kaggle/input/mimic-cxr-jpg-2-0-0/files/p10/p10000032/s50414267/02aa804e-bde0afdd-112c0b34-7bc16630-4e384014.jpg
DATA_ROOT = Path("/kaggle/input/mimic-cxr-jpg-2-0-0")
FILES_ROOT = DATA_ROOT / "files"
METADATA_CSV = DATA_ROOT / "mimic-cxr-2.0.0-metadata.csv.gz"
REPORTS_CSV = DATA_ROOT / "mimic-cxr-2.0.0-reports.csv.gz"

SUBSET_SIZE = 2000
SEED = 42
np.random.seed(SEED)

# -----------------------------
# Report parsing helpers
# -----------------------------
def _extract_section(report_text: str, section_name: str) -> str:
    """
    Extract a section from free-text radiology report.

    Why this matters clinically:
    - Reports are semi-structured; Findings and Impression are often present as headers.
    - Impression is high-level diagnostic synthesis; Findings captures imaging evidence.
    - Training on both improves factual grounding and clinically useful outputs.
    """
    if not isinstance(report_text, str) or not report_text.strip():
        return ""

    # Match headings like "FINDINGS:", "Findings:", with robust stopping at next heading.
    pattern = rf"(?is)\b{section_name}\b\s*:\s*(.*?)(?=\n\s*[A-Z][A-Z\s]{2,}:|\Z)"
    m = re.search(pattern, report_text)
    if not m:
        return ""

    section = re.sub(r"\s+", " ", m.group(1)).strip()
    return section


def extract_findings_impression(report_text: str) -> Tuple[str, str]:
    findings = _extract_section(report_text, "findings")
    impression = _extract_section(report_text, "impression")

    # Fallback heuristic if explicit sections are absent.
    # We split around common impression cue and keep robust defaults.
    if not findings and not impression and isinstance(report_text, str):
        txt = re.sub(r"\s+", " ", report_text).strip()
        parts = re.split(r"(?i)\bimpression\b\s*:\s*", txt, maxsplit=1)
        if len(parts) == 2:
            findings = parts[0].strip(" :")
            impression = parts[1].strip(" :")
        else:
            findings = txt[:1200]
            impression = txt[:400]

    return findings, impression


def build_mimic_jpg_path(subject_id: int, study_id: int, dicom_id: str, files_root: Path) -> Path:
    """
    Build expected MIMIC-CXR-JPG path from IDs.
    Pattern: files/pXX/pSUBJECT/sSTUDY/DICOM.jpg
    """
    subject_str = f"p{int(subject_id)}"
    prefix = subject_str[:3]  # e.g., p10 from p10000032
    return files_root / prefix / subject_str / f"s{int(study_id)}" / f"{dicom_id}.jpg"


# -----------------------------
# Load and join metadata + reports
# -----------------------------
assert METADATA_CSV.exists(), f"Missing metadata file: {METADATA_CSV}"
assert REPORTS_CSV.exists(), f"Missing reports file: {REPORTS_CSV}"
assert FILES_ROOT.exists(), f"Missing files directory: {FILES_ROOT}"

meta_df = pd.read_csv(METADATA_CSV, compression="gzip")
rep_df = pd.read_csv(REPORTS_CSV, compression="gzip")

# Merge on patient + study keys (many DICOMs per study)
merged = meta_df.merge(rep_df[["subject_id", "study_id", "report"]], on=["subject_id", "study_id"], how="inner")

# Extract clinically relevant sections
merged[["findings", "impression"]] = merged["report"].apply(lambda x: pd.Series(extract_findings_impression(x)))

# Keep examples where we have at least one useful section
merged = merged[(merged["findings"].str.len() > 0) | (merged["impression"].str.len() > 0)].copy()

# Create local JPG paths per DICOM
merged["image_path"] = merged.apply(
    lambda r: str(build_mimic_jpg_path(r["subject_id"], r["study_id"], r["dicom_id"], FILES_ROOT)), axis=1
)

# Keep only rows where image file exists
merged["exists"] = merged["image_path"].map(os.path.exists)
merged = merged[merged["exists"]].copy()

# Build target report text with strict structure for instruction tuning
merged["target_report"] = merged.apply(
    lambda r: f"Findings: {r['findings'].strip()}\nImpression: {r['impression'].strip()}".strip(), axis=1
)

# Small subset for hackathon-friendly training runtime
subset_df = merged.sample(n=min(SUBSET_SIZE, len(merged)), random_state=SEED).reset_index(drop=True)

print(f"Metadata rows: {len(meta_df):,}")
print(f"Reports rows: {len(rep_df):,}")
print(f"Merged valid rows: {len(merged):,}")
print(f"Training subset rows: {len(subset_df):,}")

display(subset_df[["subject_id", "study_id", "dicom_id", "image_path", "findings", "impression"]].head(3))

## Phase 2: 4-Bit Model Loading & LoRA Configuration

In [ ]:
# Code Cell 3: Load PaliGemma in 4-bit + initialize processor

import torch
from transformers import AutoProcessor, PaliGemmaForConditionalGeneration, BitsAndBytesConfig

MODEL_ID = "google/paligemma-3b-pt-224"

# QLoRA core trick:
# Load base model weights in 4-bit NF4 quantization to slash VRAM usage.
# This keeps memory low while LoRA adapters learn task-specific deltas.
compute_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

processor = AutoProcessor.from_pretrained(MODEL_ID)

model = PaliGemmaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=compute_dtype,
    device_map="auto",
)

# Ensure tokenizer has a pad token for stable batching.
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

print(f"Loaded model: {MODEL_ID}")
print(f"Trainable params before LoRA: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
# Code Cell 4: Configure LoRA adapters and freeze non-target modules

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare model for low-bit training stability (casts layer norms and enables gradient checkpoint friendliness).
model = prepare_model_for_kbit_training(model)

# Freeze the vision encoder so adaptation budget goes into multimodal-language behavior.
# Clinical rationale: for report generation, language grounding and cross-modal fusion often need
# more adaptation than low-level visual feature extraction when starting from strong pretrained vision backbones.
for name, param in model.named_parameters():
    if "vision_tower" in name or "vision_model" in name:
        param.requires_grad = False

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

# Optional safety freeze for shared embeddings if memory constrained
# for p in model.language_model.model.embed_tokens.parameters():
#     p.requires_grad = False

model.print_trainable_parameters()

# Extra transparency: list a few trainable tensors
trainable_names = [n for n, p in model.named_parameters() if p.requires_grad]
print("Sample trainable parameter names:")
for n in trainable_names[:20]:
    print(" -", n)

## Phase 3: Multimodal Training Loop

In [ ]:
# Code Cell 5: Custom Dataset + collator for multimodal report tuning

from dataclasses import dataclass
from typing import Dict, List

import torch
from torch.utils.data import Dataset

INSTRUCTION_PROMPT = (
    "You are a radiology assistant. Generate a structured chest X-ray report with exactly two sections: "
    "Findings and Impression."
)

class MimicCXRReportDataset(Dataset):
    """
    Returns multimodal training examples for instruction tuning.

    Why this shaping works:
    - Image provides visual evidence.
    - Instruction prompt standardizes behavior.
    - Target text forces consistent clinical report schema.
    """

    def __init__(self, df: pd.DataFrame, processor, image_size: int = 224, max_length: int = 512):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.image_size = image_size
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        row = self.df.iloc[idx]

        image = Image.open(row["image_path"]).convert("RGB")
        image = image.resize((self.image_size, self.image_size), Image.Resampling.BICUBIC)

        # We concatenate instruction + target report.
        # For causal LM fine-tuning, labels mirror input_ids (pad tokens masked in collator).
        target_text = row["target_report"]
        full_text = f"{INSTRUCTION_PROMPT}\n{target_text}"

        encoded = self.processor(
            images=image,
            text=full_text,
            padding=False,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )

        item = {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "pixel_values": encoded["pixel_values"].squeeze(0),
        }
        return item


@dataclass
class CXRDataCollator:
    processor: AutoProcessor

    def __call__(self, features: List[Dict[str, torch.Tensor]]) -> Dict[str, torch.Tensor]:
        # Dynamic token padding keeps batches memory-efficient.
        input_ids = [f["input_ids"] for f in features]
        attention_mask = [f["attention_mask"] for f in features]
        pixel_values = torch.stack([f["pixel_values"] for f in features])

        batch_tokens = self.processor.tokenizer.pad(
            {"input_ids": input_ids, "attention_mask": attention_mask},
            return_tensors="pt",
        )

        labels = batch_tokens["input_ids"].clone()
        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        return {
            "input_ids": batch_tokens["input_ids"],
            "attention_mask": batch_tokens["attention_mask"],
            "pixel_values": pixel_values,
            "labels": labels,
        }


# Split for train/eval demonstration
train_frac = 0.9
split_idx = int(len(subset_df) * train_frac)
train_df = subset_df.iloc[:split_idx].reset_index(drop=True)
eval_df = subset_df.iloc[split_idx:].reset_index(drop=True)

train_dataset = MimicCXRReportDataset(train_df, processor=processor)
eval_dataset = MimicCXRReportDataset(eval_df, processor=processor)
collator = CXRDataCollator(processor=processor)

print(f"Train samples: {len(train_dataset):,}")
print(f"Eval samples: {len(eval_dataset):,}")

In [ ]:
# Code Cell 6: TrainingArguments for QLoRA fine-tuning

from transformers import TrainingArguments

OUTPUT_DIR = "./paligemma-mimiccxr-qlora"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,  # Effective batch improves stability on limited VRAM.
    learning_rate=2e-5,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=20,
    eval_steps=200,
    save_steps=200,
    evaluation_strategy="steps",
    save_strategy="steps",
    save_total_limit=2,
    fp16=(compute_dtype == torch.float16),
    bf16=(compute_dtype == torch.bfloat16),
    optim="paged_adamw_8bit",  # Memory-efficient optimizer for QLoRA workflows.
    gradient_checkpointing=True,
    dataloader_num_workers=2,
    report_to="none",
    remove_unused_columns=False,
)

print(training_args)

In [ ]:
# Code Cell 7: Trainer initialization and fine-tuning

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset if len(eval_dataset) > 0 else None,
    data_collator=collator,
)

train_result = trainer.train()
print(train_result)

# Save LoRA adapter and processor artifacts.
trainer.model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)

print(f"Saved fine-tuned adapter artifacts to: {OUTPUT_DIR}")

## Phase 4: Clinical Inference & Metrics

In [ ]:
# Code Cell 8: Inference function for real MIMIC-CXR JPG

import textwrap

@torch.no_grad()
def generate_cxr_report(image_path: str, max_new_tokens: int = 256) -> str:
    """
    Generate a structured report from a chest X-ray image.

    Clinical note:
    - Keep generation deterministic and concise for safer draft reports.
    - This output is a draft and must be reviewed by a qualified radiologist.
    """
    model.eval()

    image = Image.open(image_path).convert("RGB").resize((224, 224), Image.Resampling.BICUBIC)
    prompt = (
        "You are a radiology assistant. Generate a structured chest X-ray report with exactly two sections: "
        "Findings and Impression."
    )

    inputs = processor(images=image, text=prompt, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    generated_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        num_beams=1,
        temperature=1.0,
        repetition_penalty=1.05,
    )

    decoded = processor.tokenizer.decode(generated_ids[0], skip_special_tokens=True)

    # Strip echoed prompt when present.
    if decoded.startswith(prompt):
        decoded = decoded[len(prompt):].strip()

    # Enforce structure if model returns free text.
    if "Findings:" not in decoded:
        decoded = f"Findings: {decoded}\nImpression: Review required."
    elif "Impression:" not in decoded:
        decoded = decoded + "\nImpression: Review required."

    return decoded

# Smoke test candidate (first eval sample)
test_image_path = eval_df.iloc[0]["image_path"] if len(eval_df) > 0 else train_df.iloc[0]["image_path"]
print("Test image:", test_image_path)
print(textwrap.shorten(generate_cxr_report(test_image_path), width=240, placeholder=" ..."))

In [ ]:
# Code Cell 9: Visual result panel (X-ray + ground truth + generated report)

import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

def plot_case_comparison(row: pd.Series):
    image = Image.open(row["image_path"]).convert("L")
    generated = generate_cxr_report(row["image_path"])
    gt = row["target_report"]

    fig = plt.figure(figsize=(16, 6))
    gs = GridSpec(1, 2, width_ratios=[1, 1.5], figure=fig)

    ax_img = fig.add_subplot(gs[0, 0])
    ax_txt = fig.add_subplot(gs[0, 1])

    ax_img.imshow(image, cmap="gray")
    ax_img.set_title(f"MIMIC-CXR\nsubject_id={row['subject_id']} study_id={row['study_id']}")
    ax_img.axis("off")

    text_block = (
        "GROUND TRUTH REPORT\n"
        + "-" * 30
        + f"\n{gt}\n\n"
        + "MODEL GENERATED REPORT\n"
        + "-" * 30
        + f"\n{generated}"
    )
    ax_txt.text(0.01, 0.99, text_block, va="top", ha="left", fontsize=10, family="monospace", wrap=True)
    ax_txt.axis("off")

    plt.tight_layout()
    plt.show()

# Visualize one held-out example
demo_row = eval_df.iloc[0] if len(eval_df) > 0 else train_df.iloc[0]
plot_case_comparison(demo_row)

In [ ]:
# Code Cell 10 (Optional): BLEU + ROUGE quantitative evaluation

from tqdm.auto import tqdm
import evaluate

# Load metrics (can take a moment first time)
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")

def evaluate_generation(df: pd.DataFrame, max_samples: int = 100) -> pd.DataFrame:
    sample_df = df.sample(n=min(max_samples, len(df)), random_state=SEED).reset_index(drop=True)

    predictions = []
    references = []

    for _, row in tqdm(sample_df.iterrows(), total=len(sample_df)):
        pred = generate_cxr_report(row["image_path"])
        ref = row["target_report"]
        predictions.append(pred)
        references.append(ref)

    rouge_scores = rouge.compute(predictions=predictions, references=references)

    # BLEU expects tokenized refs as list of list
    bleu_scores = bleu.compute(
        predictions=[p.split() for p in predictions],
        references=[[r.split()] for r in references],
    )

    metrics = {
        "rouge1": rouge_scores.get("rouge1", None),
        "rouge2": rouge_scores.get("rouge2", None),
        "rougeL": rouge_scores.get("rougeL", None),
        "bleu": bleu_scores.get("bleu", None),
        "n_samples": len(sample_df),
    }

    return pd.DataFrame([metrics])

metrics_df = evaluate_generation(eval_df if len(eval_df) > 0 else train_df, max_samples=64)
display(metrics_df)